Part A - Write a function generate_housing_data(N, true_w, noise_std, seed) that:

Creates X of shape (N, len(true_w)) with random feature values (e.g. np.random.randn)
Computes y = X @ true_w + noise, where noise is drawn from a normal distribution with std noise_std
Returns X, y

Generate a dataset with N=200, true_w = np.array([3.0, -1.5]) (representing something like "price per sqft" and "price penalty per mile from city center"), noise_std=1.0.





Part B (Train/Test Split) - Manually slice (no sklearn train_test_split yet): first 160 rows train, last 40 test.




Part C (Fit and Compare) - Fit linear_regression_closed_form on training set only. Compare learned weights to true_w.




Part D (Evaluate Properly) - Compute train MSE and test MSE separately. Compare. Then retrain on all 200 points, compare training MSE to the original 160-point training MSE.




Part E (Bag-of-Words) - Given a small labeled reviews/labels dataset (6 reviews, positive/negative), build vocab, featurize with bag_of_words, fit closed-form regression, predict + round to 0/1, compute accuracy. Answer: where does linear regression show strain at a classification task? (sets up Ch4 motivation)


In [ ]:
##Final Solution
import numpy as np
from sklearn.linear_model import LinearRegression
##currently don't know y, but already have true_w, so generate x and noise, and derive y from y = (X*true_W) + noise
def generate_housing_data(N, true_w, noise_std, seed):
##np.random.seed(seed) gurantees that X and noise are randomly generated once, and then produces the same output every time after that (mantains consistency when the function gets called repeatedly)
  np.random.seed(seed)
  X = np.random.randn(N, len(true_w))
##draws N independent samples from a normal distrubution with mean = 0, standard deviation = noise_std
  noise = np.random.randn(N)*noise_std
  Y = X@true_w +noise
  return X, Y
X, Y = generate_housing_data(200, np.array([3.0,-1.5]), 1.0, 42)
##Part B
X_train = X[0:160, :]
X_test = X[160:200, :]
Y_train = Y[0:160]
Y_test = Y[160:200]
##Part C
def linear_regression_closed_form(X, Y):
  return np.linalg.solve(X.T @ X, X.T @ Y)
learned_w = linear_regression_closed_form(X_train, Y_train)
print(learned_w)
##Part D
Y_train_predict = X_train @ learned_w
Y_test_predict = X_test @ learned_w
train_MSE = np.mean((Y_train - Y_train_predict)**2)
test_MSE = np.mean((Y_test - Y_test_predict)**2)
learned_w_full = linear_regression_closed_form(X, Y)
Y_total_predict = X @ learned_w_full
total_MSE = np.mean((Y - Y_total_predict)**2)
print(train_MSE, test_MSE, total_MSE)
##Part E
reviews = [
    "great product loved it",
    "terrible waste of money",
    "excellent quality highly recommend",
    "awful do not buy",
    "amazing value great price",
    "poor quality very disappointed"
]
vocab = set()
for review in reviews:
  for word in review.lower().split():
    vocab.add(word)
def bag_of_words(reviews, vocab):
  rows = []
  for review in reviews:
    count = {word:0 for word in vocab}
    for word in review.lower().split():
      if word in vocab:
        count[word]+=1
  ##row inputs the vocab count accumulated for that singular review, while rows.append adds the row you just built to a matrix that in the end while hold the vocab count for each review, and then you go to the next review
    row = [count[word] for word in vocab]
    rows.append(row)
  return np.array(rows, dtype = float)
vocab = sorted(vocab)
labels = np.array([1, 0, 1, 0, 1, 0])
X_bow = bag_of_words(reviews, vocab)
##can't use closed form linear regression because X isn't invertible
model = LinearRegression()
model.fit(X_bow, labels)
predictions = model.predict(X_bow)
predictions_rounded = np.round(predictions)
prediction_accuracy = np.mean((labels==predictions_rounded))
print(prediction_accuracy)


[ 3.09745077 -1.4647255 ]
0.9788660807403226 1.014694828371585 0.9850254374880673
1.0
